# GIAnT LoCo vs SILo validation

This notebook validates and compares `experiment_summary.h5` outputs produced by GIAnT using
`summarize_LoCo` and `SILo`.

The analysis is intentionally organized in this order:

1. **HDF5/schema integrity** — make sure each file is internally self-consistent.
2. **Input equivalence** — verify that both methods saw the same registered imaging data.
3. **Stored-data QC** — frame rejection, finite values, source profiles, baselines, SNR.
4. **Activity-image comparison** — direct human-readable comparison of `act_im`.
5. **Detected-source comparison** — matched, LoCo-only, and SILo-only source locations.
6. **Matched-source footprints and traces** — inspect whether apparently corresponding sources contain the same signal.
7. **Unmatched-source galleries** — directly inspect puncta detected by only one pathway.

### Important analysis policy

No temporal filtering, smoothing, high-pass/low-pass processing, or event-triggered transformation is applied.
Stored traces are plotted directly. Image percentile limits and z-scoring are used **only for display/comparison of static activity images** and do not alter source detection or traces.

The notebook supports both current row-major GIAnT HDF5 files and legacy MATLAB/column-major files such as the
benchmark files used to develop it.


In [ ]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# ---------------------------------------------------------------------
# USER CONFIGURATION
# ---------------------------------------------------------------------
LOCO_FILE = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+ASAP8\879442\879442_2026-08-20_08-36-59\slap2\dynamic_data\Neuron2\source_extraction\summarize_LoCo\experiment_summary.h5")
SILO_FILE = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\803496_2025-07-25_13-02-10\slap2\dynamic_data\source_extraction\experiment_summary.h5")
# Sources whose centroids are within this many pixels are considered
# spatially corresponding for the simple nearest-neighbor QC.
MATCH_RADIUS_PX = 3.0

# Display-only percentile limits for static images. No image smoothing.
IMAGE_PERCENTILES = (1.0, 99.0)

assert LOCO_FILE.exists(), f"LoCo file not found: {LOCO_FILE.resolve()}"
assert SILO_FILE.exists(), f"SILo file not found: {SILO_FILE.resolve()}"

FILES = {"LoCo": LOCO_FILE, "SILo": SILO_FILE}
print({k: str(v.resolve()) for k, v in FILES.items()})


In [ ]:
%matplotlib notebook

## HDF5 orientation helpers

GIAnT marks files with `/row_major`.

- `row_major == 1`: dataset dimensions follow the current README convention in `h5py`.
- `row_major == 0`: legacy MATLAB files appear axis-reversed in `h5py`.

The helper functions below return images as `(rows, cols)`, coordinates as `(sources, 3)` in
`[z, y, x]`, and traces as `(frames,)` regardless of storage convention.


In [ ]:
def _scalar(ds):
    return np.asarray(ds[()]).ravel()[0].item()

def is_row_major(f):
    return bool(_scalar(f["row_major"])) if "row_major" in f else False

def path_names(f):
    return sorted([k for k in f.keys() if k.startswith("Path")],
                  key=lambda x: int(x.replace("Path", "")))

def read_object_string(ds):
    x = ds[()]
    if ds.dtype.kind != "O":
        arr = np.asarray(x)
        if arr.size == 1:
            return arr.ravel()[0].item()
        return arr

    vals = []
    for ref in np.asarray(x).ravel():
        if isinstance(ref, h5py.Reference) and ref:
            arr = ds.file[ref][()]
            if np.asarray(arr).dtype.kind in "ui":
                vals.append("".join(map(chr, np.asarray(arr).ravel(order="F"))))
            else:
                vals.append(str(np.asarray(arr)))
        else:
            vals.append(str(ref))
    return vals[0] if len(vals) == 1 else vals

def read_params(path):
    out = {}
    with h5py.File(path, "r") as f:
        for key, obj in f["params"].items():
            if isinstance(obj, h5py.Dataset):
                out[key] = read_object_string(obj)
    return out

def get_activity_channel_zero_based(f):
    # GIAnT stores activityChannel using MATLAB 1-based channel indexing.
    return int(round(_scalar(f["params/activityChannel"]))) - 1

def read_coords(f, path_name):
    a = np.asarray(f[f"{path_name}/sources/spatial/coords"][()])
    return a if is_row_major(f) else a.T

def read_peaks(f, path_name):
    a = np.asarray(f[f"{path_name}/visualizations/act_im_peaks"][()])
    return a if is_row_major(f) else a.T

def read_act_image(f, path_name, z=0):
    a = np.asarray(f[f"{path_name}/visualizations/act_im"][()], dtype=float)
    return a[z, :, :] if is_row_major(f) else a[:, :, z].T

def read_mean_image(f, path_name, channel=None, z=0):
    if channel is None:
        channel = get_activity_channel_zero_based(f)
    a = np.asarray(f[f"{path_name}/visualizations/mean_im"][()], dtype=float)
    return a[channel, z, :, :] if is_row_major(f) else a[:, :, z, channel].T

def read_profile(f, path_name, source, z=0):
    a = np.asarray(f[f"{path_name}/sources/spatial/profiles"][()], dtype=float)
    return a[source, z, :, :] if is_row_major(f) else a[:, :, z, source].T

def read_temporal(f, path_name, field, source, channel=None):
    if channel is None:
        channel = get_activity_channel_zero_based(f)
    a = np.asarray(f[f"{path_name}/sources/temporal/{field}"][()], dtype=float)
    return a[source, channel, :] if is_row_major(f) else a[:, channel, source]

def read_snr(f, path_name):
    return np.asarray(f[f"{path_name}/sources/temporal/SNR"][()], dtype=float).ravel()

def read_frame_vector(f, path_name, field):
    return np.asarray(f[f"{path_name}/frame_info/{field}"][()]).ravel()

def read_global_f(f, path_name):
    a = np.asarray(f[f"{path_name}/global/F"][()], dtype=float)
    return a.T if is_row_major(f) else a  # always frames x channels

def n_sources(f, path_name):
    return read_coords(f, path_name).shape[0]

def finite_image_mask(f, path_name):
    return np.isfinite(read_act_image(f, path_name))


## 1. File structure and parameter audit

The structural validator checks that required groups are present and that source/frame dimensions agree.
The parameter table then identifies differences between the two runs.

`maxSynapseDensity` is retained in the parameter comparison because it is relevant to LoCo, but it should
**not** be interpreted as a SILo parameter merely because it is present in a shared HDF5 parameter struct.


In [ ]:
REQUIRED = [
    "Z_depths",
    "frame_info/discard_frames",
    "frame_info/frame_line_idxs",
    "frame_info/offlineXshifts",
    "frame_info/offlineYshifts",
    "frame_info/onlineXshifts",
    "frame_info/onlineYshifts",
    "frame_info/onlineZshifts",
    "frame_info/trial_num_frames",
    "global/F",
    "sources/spatial/coords",
    "sources/spatial/profiles",
    "sources/temporal/F0",
    "sources/temporal/SNR",
    "sources/temporal/dF_denoised",
    "sources/temporal/dF_ls",
    "sources/temporal/events",
    "visualizations/act_im",
    "visualizations/act_im_peaks",
    "visualizations/mean_im",
]

def validate_file(path, label):
    rows = []
    with h5py.File(path, "r") as f:
        rows.append((label, "/", "row_major present", "row_major" in f))
        rows.append((label, "/", "params present", "params" in f))

        for pn in path_names(f):
            g = f[pn]
            for rel in REQUIRED:
                rows.append((label, pn, f"present: {rel}", rel in g))

            coords = read_coords(f, pn)
            peaks = read_peaks(f, pn)
            ns = coords.shape[0]

            trial_counts = read_frame_vector(f, pn, "trial_num_frames").astype(int)
            nf = int(trial_counts.sum())

            prof_ds = g["sources/spatial/profiles"]
            f0_ds = g["sources/temporal/F0"]
            snr = read_snr(f, pn)
            global_f = read_global_f(f, pn)
            discard = read_frame_vector(f, pn, "discard_frames")

            # orientation-independent source counts
            prof_ns = prof_ds.shape[0] if is_row_major(f) else prof_ds.shape[-1]
            f0_ns = f0_ds.shape[0] if is_row_major(f) else f0_ds.shape[-1]

            checks = {
                "coords are Nx3": coords.ndim == 2 and coords.shape[1] == 3,
                "peaks are Nx3": peaks.ndim == 2 and peaks.shape[1] == 3,
                "profiles source count": prof_ns == ns,
                "F0 source count": f0_ns == ns,
                "SNR source count": len(snr) == ns,
                "peak/source count": peaks.shape[0] == ns,
                "global/F frame count": global_f.shape[0] == nf,
                "discard frame count": len(discard) == nf,
                "finite source coordinates": np.isfinite(coords).all(),
                "finite SNR": np.isfinite(snr).all(),
            }
            for name, ok in checks.items():
                rows.append((label, pn, name, bool(ok)))

    return pd.DataFrame(rows, columns=["file", "scope", "check", "pass"])

validation = pd.concat(
    [validate_file(path, label) for label, path in FILES.items()],
    ignore_index=True,
)

display(validation.groupby("file")["pass"].agg(["count", "sum"]))
failed = validation.loc[~validation["pass"]]
display(failed if len(failed) else pd.DataFrame({"result": ["All structural checks passed."]}))


In [ ]:
PARAMS = {label: read_params(path) for label, path in FILES.items()}
all_keys = sorted(set(PARAMS["LoCo"]) | set(PARAMS["SILo"]))

def values_equal(a, b):
    if isinstance(a, np.ndarray) and isinstance(b, np.ndarray):
        return np.array_equal(a, b, equal_nan=True)
    try:
        return bool(a == b)
    except Exception:
        return False

param_rows = []
for key in all_keys:
    a = PARAMS["LoCo"].get(key)
    b = PARAMS["SILo"].get(key)
    if not values_equal(a, b):
        param_rows.append({"parameter": key, "LoCo": a, "SILo": b})

param_diff = pd.DataFrame(param_rows)
display(param_diff)


## 2. Prove that the underlying imaging inputs match

This is a critical benchmark step. A method comparison is much easier to interpret if the registered mean images,
frame indices, motion traces, and global fluorescence agree before source detection.

The code below reports exact equality where appropriate and compares `global/F` only on samples finite in both files,
because different motion thresholds may cause one run to NaN-out a few additional frames.


In [ ]:
def compare_input_equivalence(loco_path, silo_path):
    rows = []
    with h5py.File(loco_path, "r") as fl, h5py.File(silo_path, "r") as fs:
        common_paths = sorted(set(path_names(fl)) & set(path_names(fs)),
                              key=lambda x: int(x.replace("Path", "")))

        for pn in common_paths:
            # Static mean images, all channels
            n_channels = read_global_f(fl, pn).shape[1]
            mean_exact = all(
                np.array_equal(read_mean_image(fl, pn, ch), read_mean_image(fs, pn, ch), equal_nan=True)
                for ch in range(n_channels)
            )

            rows.append({"path": pn, "quantity": "mean_im (all channels)", "exact_on_comparable_values": mean_exact})

            for field in [
                "frame_line_idxs",
                "offlineXshifts", "offlineYshifts",
                "onlineXshifts", "onlineYshifts", "onlineZshifts",
                "trial_num_frames",
            ]:
                a = read_frame_vector(fl, pn, field)
                b = read_frame_vector(fs, pn, field)
                rows.append({
                    "path": pn,
                    "quantity": field,
                    "exact_on_comparable_values": np.array_equal(a, b, equal_nan=True),
                })

            a = read_global_f(fl, pn)
            b = read_global_f(fs, pn)
            finite = np.isfinite(a) & np.isfinite(b)
            rows.append({
                "path": pn,
                "quantity": "global/F on common finite samples",
                "exact_on_comparable_values": np.array_equal(a[finite], b[finite]),
            })

    return pd.DataFrame(rows)

input_equivalence = compare_input_equivalence(LOCO_FILE, SILO_FILE)
display(input_equivalence)


## 3. Frame rejection and stored-data QC

Useful checks here include:

- fraction of frames discarded,
- whether temporal NaNs occur exactly where frames were discarded,
- positive/finite `F0`,
- finite SNR,
- normalized and localized source footprints,
- source-count differences between methods.

A lower source count with a higher median SNR is often a signature of a more conservative detector, but this notebook
does not assume that either detector is "better" from that fact alone.


In [ ]:
def source_profile_matrix(f, pn):
    # Return rows x cols x sources
    ns = n_sources(f, pn)
    return np.stack([read_profile(f, pn, s) for s in range(ns)], axis=-1)

qc_rows = []
with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
    for method, f in [("LoCo", fl), ("SILo", fs)]:
        for pn in path_names(f):
            ns = n_sources(f, pn)
            discard = read_frame_vector(f, pn, "discard_frames").astype(bool)
            snr = read_snr(f, pn)
            f0 = np.stack([read_temporal(f, pn, "F0", s) for s in range(ns)], axis=1)
            profiles = source_profile_matrix(f, pn)

            # Check NaNs in stored least-squares traces.
            dF = np.stack([read_temporal(f, pn, "dF_ls", s) for s in range(ns)], axis=1)
            any_nan_frame = np.isnan(dF).any(axis=1)

            profile_sums = np.nansum(profiles, axis=(0, 1))
            negative_profile_pixels = int(np.sum(profiles < -1e-8))

            qc_rows.append({
                "method": method,
                "path": pn,
                "sources": ns,
                "frames": len(discard),
                "discard_frames": int(discard.sum()),
                "discard_pct": 100.0 * discard.mean(),
                "dF_nan_frames_equal_discard": bool(np.array_equal(any_nan_frame, discard)),
                "SNR_min": float(np.min(snr)),
                "SNR_median": float(np.median(snr)),
                "SNR_max": float(np.max(snr)),
                "F0_min": float(np.nanmin(f0)),
                "F0_finite_fraction": float(np.isfinite(f0).mean()),
                "profile_sum_error_max": float(np.max(np.abs(profile_sums - 1.0))),
                "profile_negative_pixels_lt_-1e-8": negative_profile_pixels,
            })

qc = pd.DataFrame(qc_rows)
display(qc)


In [ ]:
# Compare discard-frame sets directly.
discard_rows = []
with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
    for pn in sorted(set(path_names(fl)) & set(path_names(fs)),
                     key=lambda x: int(x.replace("Path", ""))):
        dl = read_frame_vector(fl, pn, "discard_frames").astype(bool)
        ds = read_frame_vector(fs, pn, "discard_frames").astype(bool)
        discard_rows.append({
            "path": pn,
            "LoCo_discard": int(dl.sum()),
            "SILo_discard": int(ds.sum()),
            "both": int((dl & ds).sum()),
            "LoCo_only": int((dl & ~ds).sum()),
            "SILo_only": int((ds & ~dl).sum()),
        })

display(pd.DataFrame(discard_rows))


## 4. Activity-image comparison

The two methods can store `act_im` on very different absolute scales. Therefore:

- the first row uses **separate percentile display limits** for each method;
- the second row standardizes valid pixels within each activity image solely to compare spatial structure.

No spatial filtering or smoothing is performed.

The quantitative table reports Pearson/Spearman spatial correlations and the Jaccard overlap of the top 1% of
activity-image pixels.


In [ ]:
def standardize_valid_image(im):
    out = np.full_like(im, np.nan, dtype=float)
    mask = np.isfinite(im)
    vals = im[mask]
    if vals.size and np.std(vals) > 0:
        out[mask] = (vals - np.mean(vals)) / np.std(vals)
    return out

def activity_image_metrics(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    av = a[mask]
    bv = b[mask]

    pearson = np.corrcoef(av, bv)[0, 1]
    spearman = spearmanr(av, bv).statistic

    ta = np.percentile(av, 99)
    tb = np.percentile(bv, 99)
    A = av >= ta
    B = bv >= tb
    union = np.sum(A | B)
    jaccard = np.sum(A & B) / union if union else np.nan

    return pearson, spearman, jaccard

def show_activity_images(path_name):
    with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
        mean_im = read_mean_image(fl, path_name)
        loco = read_act_image(fl, path_name)
        silo = read_act_image(fs, path_name)

    zl = standardize_valid_image(loco)
    zs = standardize_valid_image(silo)
    zdiff = zs - zl

    fig, ax = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)

    lo, hi = np.nanpercentile(mean_im, IMAGE_PERCENTILES)
    ax[0, 0].imshow(mean_im, cmap="gray", vmin=lo, vmax=hi)
    ax[0, 0].set_title(f"{path_name}: mean image")

    for a, im, title in [
        (ax[0, 1], loco, "LoCo act_im"),
        (ax[0, 2], silo, "SILo act_im"),
    ]:
        valid = np.isfinite(im)
        lim = np.nanpercentile(np.abs(im[valid]), 99)
        a.imshow(im, cmap="coolwarm", vmin=-lim, vmax=lim)
        a.set_title(f"{title}\n(separate display scale)")

    ax[1, 0].imshow(zl, cmap="coolwarm", vmin=-3, vmax=3)
    ax[1, 0].set_title("LoCo act_im standardized")
    ax[1, 1].imshow(zs, cmap="coolwarm", vmin=-3, vmax=3)
    ax[1, 1].set_title("SILo act_im standardized")

    dlim = np.nanpercentile(np.abs(zdiff), 99)
    ax[1, 2].imshow(zdiff, cmap="bwr", vmin=-dlim, vmax=dlim)
    ax[1, 2].set_title("SILo z - LoCo z")

    for a in ax.ravel():
        a.axis("off")

    plt.show()

    pearson, spearman, jaccard = activity_image_metrics(loco, silo)
    return pd.DataFrame([{
        "path": path_name,
        "Pearson_r": pearson,
        "Spearman_rho": spearman,
        "top_1pct_pixel_Jaccard": jaccard,
    }])

activity_metrics = []
with h5py.File(LOCO_FILE, "r") as f:
    pns = path_names(f)

for pn in pns:
    activity_metrics.append(show_activity_images(pn))

display(pd.concat(activity_metrics, ignore_index=True))


## 5. Detected-source locations: matched vs method-only puncta

Two complementary matching views are useful:

1. **Symmetric nearest-neighbor distance** — best for identifying whether each source has any nearby source in the other method.
   This permits one source to be near multiple sources, which is useful for detecting apparent splits/merges.
2. **Hungarian one-to-one matching** — useful when comparing corresponding footprints and traces.

The default `MATCH_RADIUS_PX = 3` is deliberately exposed as a configuration variable. Inspect the images before treating
that threshold as biological ground truth.


In [ ]:
def source_nearest_table(path_name):
    with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
        lc = read_coords(fl, path_name)[:, 1:3]  # y, x
        sc = read_coords(fs, path_name)[:, 1:3]

    D = cdist(lc, sc)

    rows = []
    for i, (y, x) in enumerate(lc):
        j = int(np.argmin(D[i]))
        rows.append({
            "method": "LoCo", "source": i, "y": y, "x": x,
            "nearest_other_source": j,
            "nearest_distance_px": float(D[i, j]),
            "within_match_radius": bool(D[i, j] <= MATCH_RADIUS_PX),
        })

    for j, (y, x) in enumerate(sc):
        i = int(np.argmin(D[:, j]))
        rows.append({
            "method": "SILo", "source": j, "y": y, "x": x,
            "nearest_other_source": i,
            "nearest_distance_px": float(D[i, j]),
            "within_match_radius": bool(D[i, j] <= MATCH_RADIUS_PX),
        })

    return pd.DataFrame(rows)

def hungarian_match_table(path_name):
    with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
        lc = read_coords(fl, path_name)[:, 1:3]
        sc = read_coords(fs, path_name)[:, 1:3]

    D = cdist(lc, sc)
    li, si = linear_sum_assignment(D)

    return pd.DataFrame({
        "LoCo_source": li,
        "SILo_source": si,
        "distance_px": D[li, si],
        "within_match_radius": D[li, si] <= MATCH_RADIUS_PX,
    })

nearest_tables = {}
hungarian_tables = {}

for pn in pns:
    nearest_tables[pn] = source_nearest_table(pn)
    hungarian_tables[pn] = hungarian_match_table(pn)

    print(f"\n{pn} nearest-neighbor source summary")
    display(
        nearest_tables[pn]
        .groupby("method")
        .agg(
            sources=("source", "count"),
            with_neighbor=("within_match_radius", "sum"),
            median_nearest_distance_px=("nearest_distance_px", "median"),
        )
    )

    print(f"{pn} one-to-one Hungarian assignments")
    display(hungarian_tables[pn])


In [ ]:
def plot_source_overlay(path_name):
    with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
        mean_im = read_mean_image(fl, path_name)
        lc = read_coords(fl, path_name)[:, 1:3]
        sc = read_coords(fs, path_name)[:, 1:3]

    lo, hi = np.nanpercentile(mean_im, IMAGE_PERCENTILES)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(mean_im, cmap="gray", vmin=lo, vmax=hi)

    ax.scatter(lc[:, 1], lc[:, 0], s=70, facecolors="none", edgecolors="tab:orange",
               linewidths=1.5, label="LoCo")
    ax.scatter(sc[:, 1], sc[:, 0], s=55, marker="+", color="tab:blue",
               linewidths=1.5, label="SILo")

    for i, (y, x) in enumerate(lc):
        ax.text(x + 2, y, f"L{i}", fontsize=8)
    for i, (y, x) in enumerate(sc):
        ax.text(x + 2, y + 4, f"S{i}", fontsize=8)

    ax.set_title(f"{path_name}: detected source centroids")
    ax.legend()
    ax.set_axis_off()
    plt.show()

for pn in pns:
    plot_source_overlay(pn)


## 6. Quantify one-to-one matched source quality

For Hungarian-assigned pairs, the next cell computes:

- centroid distance,
- spatial-profile cosine similarity,
- Pearson correlation of stored `dF_ls`,
- correlation of stored denoised traces,
- event-trace correlation,
- `F0` correlation,
- SNR from each method.

Only pairs within `MATCH_RADIUS_PX` should normally be interpreted as candidate corresponding sources.
No temporal processing is applied before the correlations.


In [ ]:
def cosine_similarity(a, b):
    a = np.nan_to_num(np.asarray(a, dtype=float).ravel(), nan=0.0)
    b = np.nan_to_num(np.asarray(b, dtype=float).ravel(), nan=0.0)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return np.nan
    return float(np.dot(a, b) / (na * nb))

def finite_corr(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3 or np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return float(np.corrcoef(a[m], b[m])[0, 1])

pair_rows = []

with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
    for pn in pns:
        for _, r in hungarian_tables[pn].iterrows():
            li = int(r["LoCo_source"])
            si = int(r["SILo_source"])

            row = {
                "path": pn,
                "LoCo_source": li,
                "SILo_source": si,
                "distance_px": float(r["distance_px"]),
                "within_match_radius": bool(r["within_match_radius"]),
                "profile_cosine": cosine_similarity(
                    read_profile(fl, pn, li),
                    read_profile(fs, pn, si),
                ),
                "LoCo_SNR": read_snr(fl, pn)[li],
                "SILo_SNR": read_snr(fs, pn)[si],
            }

            for field in ["dF_ls", "dF_denoised", "events", "F0"]:
                row[f"{field}_corr"] = finite_corr(
                    read_temporal(fl, pn, field, li),
                    read_temporal(fs, pn, field, si),
                )

            pair_rows.append(row)

pair_metrics = pd.DataFrame(pair_rows)
display(pair_metrics)

print("Candidate matched pairs only:")
display(
    pair_metrics.loc[pair_metrics["within_match_radius"]]
    .groupby("path")[[
        "distance_px", "profile_cosine", "dF_ls_corr",
        "dF_denoised_corr", "events_corr", "F0_corr",
        "LoCo_SNR", "SILo_SNR"
    ]]
    .median()
)


## 7. Inspect individual matched source footprints and traces

Use `plot_source_pair(...)` on any candidate pair from `pair_metrics`.

The trace is the stored activity-channel signal, sampled directly from the HDF5 file. Set `signal` to
`"dF_ls"`, `"dF_denoised"`, `"events"`, or `"F0"`. `start_s` and `duration_s` only crop the displayed time interval.


In [ ]:
def plot_source_pair(path_name, loco_source, silo_source,
                     signal="dF_ls", start_s=0.0, duration_s=30.0):
    with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
        lp = read_profile(fl, path_name, loco_source)
        sp = read_profile(fs, path_name, silo_source)
        lt = read_temporal(fl, path_name, signal, loco_source)
        st = read_temporal(fs, path_name, signal, silo_source)

        hz_l = float(_scalar(fl["params/analyzeHz"]))
        hz_s = float(_scalar(fs["params/analyzeHz"]))

    if not np.isclose(hz_l, hz_s):
        raise ValueError(f"Different analyzeHz values: LoCo={hz_l}, SILo={hz_s}")
    hz = hz_l

    i0 = max(0, int(round(start_s * hz)))
    i1 = min(len(lt), int(round((start_s + duration_s) * hz)))
    t = np.arange(i0, i1) / hz

    fig, ax = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)

    ax[0].imshow(lp, cmap="gray")
    ax[0].set_title(f"LoCo profile L{loco_source}")
    ax[0].axis("off")

    ax[1].imshow(sp, cmap="gray")
    ax[1].set_title(f"SILo profile S{silo_source}")
    ax[1].axis("off")

    ax[2].plot(t, lt[i0:i1], linewidth=0.8, label=f"LoCo L{loco_source}")
    ax[2].plot(t, st[i0:i1], linewidth=0.8, label=f"SILo S{silo_source}")
    ax[2].set_xlabel("Time (s)")
    ax[2].set_ylabel(signal)
    ax[2].set_title(f"{path_name}: stored {signal}, no smoothing")
    ax[2].legend()

    plt.show()

# Example: automatically choose the closest matched pair in the first path.
_first = pair_metrics.loc[pair_metrics["within_match_radius"]].sort_values("distance_px").iloc[0]
plot_source_pair(
    _first["path"],
    int(_first["LoCo_source"]),
    int(_first["SILo_source"]),
    signal="dF_ls",
    start_s=0,
    duration_s=300,
)


## 8. Directly inspect method-only puncta

This is the most useful human-readable benchmark.

A source is called "method-only" here when its nearest centroid in the other method is farther than
`MATCH_RADIUS_PX`. The gallery shows the **same spatial crop** in:

1. registered mean image,
2. LoCo activity image,
3. SILo activity image,
4. the source footprint from the method that detected it.

Because `act_im` absolute scales differ, each crop is displayed with a local method-specific percentile scale.
No smoothing is applied.

This lets you distinguish two important cases:

- **activity exists in both maps but only one detector emits a source** → source selection/segmentation difference;
- **activity itself is conspicuous in only one activity map** → earlier localization-stage difference.


In [ ]:
def crop_around(im, y, x, radius=20):
    y = int(round(y))
    x = int(round(x))
    y0, y1 = max(0, y-radius), min(im.shape[0], y+radius+1)
    x0, x1 = max(0, x-radius), min(im.shape[1], x+radius+1)
    return im[y0:y1, x0:x1], (y0, y1, x0, x1)

def _display_limits(im, symmetric=False):
    vals = im[np.isfinite(im)]
    if vals.size == 0:
        return (None, None)
    if symmetric:
        lim = np.percentile(np.abs(vals), 99)
        return (-lim, lim)
    return tuple(np.percentile(vals, IMAGE_PERCENTILES))

def plot_unmatched_gallery(path_name, method, radius=20, max_sources=None):
    nt = nearest_tables[path_name]
    unique = nt[(nt["method"] == method) & (~nt["within_match_radius"])].copy()

    if max_sources is not None:
        unique = unique.head(max_sources)

    if unique.empty:
        print(f"No {method}-only sources in {path_name} at MATCH_RADIUS_PX={MATCH_RADIUS_PX}.")
        return

    with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
        mean_im = read_mean_image(fl, path_name)
        loco_act = read_act_image(fl, path_name)
        silo_act = read_act_image(fs, path_name)

        fig, axes = plt.subplots(len(unique), 4, figsize=(14, 3.5 * len(unique)), squeeze=False)

        for row_idx, (_, r) in enumerate(unique.iterrows()):
            src = int(r["source"])
            y, x = float(r["y"]), float(r["x"])

            mean_crop, bounds = crop_around(mean_im, y, x, radius)
            loco_crop, _ = crop_around(loco_act, y, x, radius)
            silo_crop, _ = crop_around(silo_act, y, x, radius)

            source_file = fl if method == "LoCo" else fs
            prof = read_profile(source_file, path_name, src)
            prof_crop, _ = crop_around(prof, y, x, radius)

            mlo, mhi = _display_limits(mean_crop)
            llo, lhi = _display_limits(loco_crop, symmetric=True)
            slo, shi = _display_limits(silo_crop, symmetric=True)

            axes[row_idx, 0].imshow(mean_crop, cmap="gray", vmin=mlo, vmax=mhi)
            axes[row_idx, 1].imshow(loco_crop, cmap="coolwarm", vmin=llo, vmax=lhi)
            axes[row_idx, 2].imshow(silo_crop, cmap="coolwarm", vmin=slo, vmax=shi)
            axes[row_idx, 3].imshow(prof_crop, cmap="gray")

            cy = y - bounds[0]
            cx = x - bounds[2]
            for a in axes[row_idx, :3]:
                a.scatter([cx], [cy], s=60, facecolors="none", edgecolors="yellow", linewidths=1.5)

            axes[row_idx, 0].set_title(f"{method} {src}: mean image")
            axes[row_idx, 1].set_title("LoCo act_im")
            axes[row_idx, 2].set_title("SILo act_im")
            axes[row_idx, 3].set_title(f"{method} source profile")

            for a in axes[row_idx]:
                a.axis("off")

        fig.suptitle(
            f"{path_name}: {method}-only sources "
            f"(nearest other source > {MATCH_RADIUS_PX:g} px)",
            y=1.01,
        )
        plt.tight_layout()
        plt.show()

for pn in pns:
    plot_unmatched_gallery(pn, "LoCo", radius=20)
    plot_unmatched_gallery(pn, "SILo", radius=20)


## 9. Rank activity at method-only source locations

A useful quantitative complement to the galleries is the standardized local activity-image value around each
method-only source. If a LoCo-only source lies on a strong SILo activity-map hotspot (or vice versa), the discrepancy
is probably downstream of activity-image generation.

The reported z values are computed over valid pixels of each static `act_im` and are used only as a spatial comparison metric.


In [ ]:
def local_max(im, y, x, radius=2):
    y = int(round(y))
    x = int(round(x))
    y0, y1 = max(0, y-radius), min(im.shape[0], y+radius+1)
    x0, x1 = max(0, x-radius), min(im.shape[1], x+radius+1)
    return float(np.nanmax(im[y0:y1, x0:x1]))

unmatched_activity_rows = []

with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
    for pn in pns:
        zl = standardize_valid_image(read_act_image(fl, pn))
        zs = standardize_valid_image(read_act_image(fs, pn))

        for _, r in nearest_tables[pn].loc[~nearest_tables[pn]["within_match_radius"]].iterrows():
            unmatched_activity_rows.append({
                "path": pn,
                "method_only_source": r["method"],
                "source": int(r["source"]),
                "nearest_other_distance_px": float(r["nearest_distance_px"]),
                "LoCo_local_max_z": local_max(zl, r["y"], r["x"]),
                "SILo_local_max_z": local_max(zs, r["y"], r["x"]),
            })

unmatched_activity = pd.DataFrame(unmatched_activity_rows)
display(unmatched_activity)


## 10. Compact benchmark summary

Use this table when comparing runs across parameter sweeps or future GIAnT versions.

For a stricter algorithmic benchmark, keep acquisition/motion-correction inputs identical and explicitly record
method-specific parameters such as LoCo `maxSynapseDensity` and any source-selection thresholds.


In [ ]:
summary_rows = []

with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:
    for pn in pns:
        loco_n = n_sources(fl, pn)
        silo_n = n_sources(fs, pn)

        nt = nearest_tables[pn]
        loco_match = int(
            nt[(nt["method"] == "LoCo")]["within_match_radius"].sum()
        )
        silo_match = int(
            nt[(nt["method"] == "SILo")]["within_match_radius"].sum()
        )

        la = read_act_image(fl, pn)
        sa = read_act_image(fs, pn)
        pearson, spearman, jaccard = activity_image_metrics(la, sa)

        summary_rows.append({
            "path": pn,
            "LoCo_sources": loco_n,
            "SILo_sources": silo_n,
            "LoCo_sources_with_neighbor": loco_match,
            "SILo_sources_with_neighbor": silo_match,
            "activity_image_Pearson_r": pearson,
            "activity_image_Spearman_rho": spearman,
            "top_1pct_pixel_Jaccard": jaccard,
        })

benchmark_summary = pd.DataFrame(summary_rows)
display(benchmark_summary)

# Optional exports for tracking parameter sweeps:
# benchmark_summary.to_csv("LoCo_vs_SILo_benchmark_summary.csv", index=False)
# pair_metrics.to_csv("LoCo_vs_SILo_matched_source_metrics.csv", index=False)
# unmatched_activity.to_csv("LoCo_vs_SILo_unmatched_source_activity.csv", index=False)


## Interpretation checklist

When reviewing a new pair of runs, answer these questions in order:

- Did both HDF5 files pass structural checks?
- Are `mean_im`, frame indices, motion traces, and common-valid `global/F` identical?
- Are discard-frame differences negligible or large enough to confound source detection?
- Are `F0`, SNR, and spatial profiles numerically well behaved?
- Do the activity maps have the same broad spatial structure?
- Which puncta are matched, split/merged, LoCo-only, or SILo-only?
- For method-only puncta, is the hotspot present in both `act_im` maps?
- For matched puncta, do footprints and unsmoothed stored traces agree?
- Are differences robust to reasonable matching-radius choices?
- If comparing algorithms rather than configured pipelines, have method-specific thresholds/density constraints been documented or swept?


In [ ]:
# =============================================================================
# Mean image + activity image overlays
# Cyan = mean fluorescence / structure
# Magenta = positive activity
# =============================================================================

def normalize_overlay_image(im, q=0.99):
    """Robustly normalize an image to [0, 1] for visualization."""
    im = np.asarray(im, dtype=float)

    finite = np.isfinite(im)
    if not finite.any():
        return np.zeros_like(im, dtype=float)

    vmax = np.nanquantile(im[finite], q)
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = np.nanmax(im[finite])

    if not np.isfinite(vmax) or vmax <= 0:
        return np.zeros_like(im, dtype=float)

    out = im / vmax
    out = np.clip(out, 0, 1)
    out[~finite] = 0

    return out


def make_mean_activity_overlay(mean_im, act_im, q_mean=0.99, q_act=0.995):
    """
    Cyan = structural mean image
    Magenta = positive activity image
    """
    mean_im = np.squeeze(np.asarray(mean_im, dtype=float))
    act_im = np.squeeze(np.asarray(act_im, dtype=float))

    # If mean image still contains a channel dimension, use channel 0
    if mean_im.ndim == 3:
        if mean_im.shape[0] <= 4:
            mean_im = mean_im[0]
        elif mean_im.shape[-1] <= 4:
            mean_im = mean_im[..., 0]
        else:
            raise ValueError(f"Could not infer mean-image channel axis: {mean_im.shape}")

    # Accommodate MATLAB/Python XY orientation differences
    if mean_im.shape != act_im.shape:
        if mean_im.T.shape == act_im.shape:
            mean_im = mean_im.T
        elif act_im.T.shape == mean_im.shape:
            act_im = act_im.T
        else:
            raise ValueError(
                f"Mean/activity image shape mismatch: "
                f"{mean_im.shape} vs {act_im.shape}"
            )

    # Structural fluorescence should be positive
    mean_pos = np.clip(mean_im, 0, None)

    # For this overlay we specifically want positive activity hotspots.
    # Negative values after local-background subtraction are not displayed.
    act_pos = np.clip(act_im, 0, None)

    mean_norm = normalize_overlay_image(mean_pos, q=q_mean)
    act_norm = normalize_overlay_image(act_pos, q=q_act)

    # Match vip-slap2-analysis summary convention:
    # mean = cyan, activity = magenta
    rgb = np.zeros((*mean_norm.shape, 3), dtype=float)

    rgb[..., 0] = act_norm                       # magenta R
    rgb[..., 1] = mean_norm                      # cyan G
    rgb[..., 2] = np.clip(mean_norm + act_norm, 0, 1)  # cyan + magenta B

    return np.clip(rgb, 0, 1)


with h5py.File(LOCO_FILE, "r") as f:
    pns = path_names(f)

fig, axes = plt.subplots(
    len(pns), 2,
    figsize=(12, 5 * len(pns)),
    squeeze=False
)

for row, path_name in enumerate(pns):

    with h5py.File(LOCO_FILE, "r") as fl, h5py.File(SILO_FILE, "r") as fs:

        # Use the same structural image for both comparisons
        mean_im = read_mean_image(fl, path_name)

        loco = read_act_image(fl, path_name)
        silo = read_act_image(fs, path_name)

    loco_overlay = make_mean_activity_overlay(
        mean_im,
        loco,
        q_mean=0.99,
        q_act=0.995,
    )

    silo_overlay = make_mean_activity_overlay(
        mean_im,
        silo,
        q_mean=0.99,
        q_act=0.995,
    )

    ax = axes[row, 0]
    ax.imshow(loco_overlay)
    ax.set_facecolor("k")
    ax.set_title(f"{path_name} — summarize_LoCo")
    ax.axis("off")

    ax = axes[row, 1]
    ax.imshow(silo_overlay)
    ax.set_facecolor("k")
    ax.set_title(f"{path_name} — SILo")
    ax.axis("off")

fig.suptitle(
    "Mean fluorescence + activity overlay\n"
    "cyan = structure, magenta = positive activity",
    y=0.97,
    fontsize=14,
)

fig.tight_layout()
plt.show()